In [19]:
import os; print(os.getcwd(), os.listdir())


/home/eek71/assignments/Final Course Project/Project ['1ibl_stackings_raw.csv', '1HNW.cif', '1hnw.out', '1ibl_hbonds_parsed.csv', '6g5h_interfaces_raw.csv', '1j5e_interfaces_raw.csv', '1hnw_hbonds_raw.csv', '6g5h_hbonds_parsed.csv', '1J5E.cif', '1j5e_hbonds_raw.csv', '1j5e.out', '6G5H.cif', '6g5h_stackings_raw.csv', '1ibl.out', '1j5e_hbonds_parsed.csv', '1hnw_hbonds_parsed.csv', 'ribosome_data.db', '1hnw_interfaces_raw.csv', '1hnw_stackings_raw.csv', '1IBL.cif', '.git', '.ipynb_checkpoints', '1j5e_stackings_raw.csv', '6g5h.out', '1ibl_hbonds_raw.csv', 'ribosome_comparison.png', '6g5h_hbonds_raw.csv', '1ibl_interfaces_raw.csv']


In [20]:
df_hbonds = pd.read_csv('1j5e_hbonds_parsed.csv')



In [21]:
!pip install biopython

Defaulting to user installation because normal site-packages is not writeable


In [25]:
import pandas as pd
import os
# Import the MMCIFParser instead of PDBParser
from Bio.PDB import MMCIFParser 

# 1. Use the exact uppercase .cif filenames from your folder
pdb_files = ['1J5E.cif', '1IBL.cif', '1HNW.cif', '6G5H.cif']
pdb_dir = 'Project/'

# 2. Initialize the correct CIF Parser
parser = MMCIFParser(QUIET=True)

all_structure_data = []

# 3. Iterative Loading and Extraction
for file in pdb_files:
    file_path = os.path.join(pdb_dir, file)
    
    if os.path.exists(file_path):
        structure_id = file.split('.')[0]
        
        # Load and initialize the CIF structure
        structure = parser.get_structure(structure_id, file_path)
        print(f"Successfully initialized: {structure_id}")
        
        # Dig into the hierarchical structure to extract coordinates
        for model in structure:
            for chain in model:
                for residue in chain:
                    # Skip water molecules (heteroatoms)
                    if residue.id[0] != ' ':
                        continue
                    for atom in residue:
                        all_structure_data.append({
                            'structure_id': structure_id,
                            'chain': chain.id,
                            'residue_name': residue.resname,
                            'residue_number': residue.id[1],
                            'atom_name': atom.name,
                            'x_coord': atom.coord[0],
                            'y_coord': atom.coord[1],
                            'z_coord': atom.coord[2],
                            'b_factor': atom.bfactor
                        })
    else:
        print(f"Warning: Could not find {file_path}")

# 4. Concatenate into a unified Master DataFrame
df_pdb_master = pd.DataFrame(all_structure_data)

print(f"\nTotal atoms extracted and concatenated: {len(df_pdb_master)}")
display(df_pdb_master.head(10))


Total atoms extracted and concatenated: 0


""


In [23]:
# Group by the structure ID and sample 2 random rows from each
display(df_pdb_master.groupby('structure_id').apply(lambda x: x.sample(2)))

KeyError: 'structure_id'

In [24]:
import sqlite3
import pandas as pd

# 1. Clean the Data: Separate Proteins from RNA (CS 210 Week 06)
# Proteins use 3-letter codes (e.g., ARG, LYS), RNA uses 1-letter codes (e.g., A, U, C, G)
df_proteins = df_pdb_master[df_pdb_master['residue_name'].str.len() == 3].copy()
df_rna = df_pdb_master[df_pdb_master['residue_name'].str.len() == 1].copy()

# Add a categorical feature for future Machine Learning grouping
df_proteins['molecule_type'] = 'Protein'
df_rna['molecule_type'] = 'RNA'

print(f"Data Cleaned! Isolated {len(df_proteins)} Protein atoms and {len(df_rna)} RNA atoms.")

# 2. Database Integration (CS 210 Week 10)
# Connect to your local SQLite database
conn = sqlite3.connect('ribosome_data.db')

# Push the cleaned DataFrames into permanent SQL tables
df_proteins.to_sql('structural_proteins', conn, if_exists='replace', index=False)
df_rna.to_sql('structural_rna', conn, if_exists='replace', index=False)

print("Successfully loaded clean 3D coordinate data into 'ribosome_data.db' tables!")

# 3. SQL Declarative Thinking: Verify the upload with a query
query = """
SELECT structure_id, molecule_type, COUNT(*) as Total_Atoms
FROM structural_proteins
GROUP BY structure_id
UNION ALL
SELECT structure_id, molecule_type, COUNT(*) as Total_Atoms
FROM structural_rna
GROUP BY structure_id
ORDER BY structure_id, molecule_type;
"""

print("\n--- SQL Query Result: Atom Counts by Molecule Type ---")
summary_df = pd.read_sql_query(query, conn)
display(summary_df)

conn.close()

KeyError: 'residue_name'

In [11]:
import sqlite3
import pandas as pd
import glob
import os

# 1. Connect to the database
conn = sqlite3.connect('ribosome_data.db')

# 2. Load H-Bond CSVs with explicit header skip and manual naming
hbond_files = glob.glob('Project/*_hbonds_parsed.csv')

if not hbond_files:
    print("ERROR: No H-bond CSV files found in 'Project/'.")
else:
    hbond_list = []
    # These are the correct headers based on the DSSR output structure
    # index: ID, nt_atom: Nucleotide Atom, aa_atom: Amino Acid Atom
    correct_headers = ['index', 'nt_atom', 'aa_atom', 'distance', 'interaction_type']
    
    for f in hbond_files:
        # Extract PDB ID from filename (e.g., '1j5e' from 'Project/1j5e_hbonds_parsed.csv')
        pdb_id = os.path.basename(f).split('_')[0].upper()
        
        # Skip the original header row and manually provide clean names
        temp_df = pd.read_csv(f, names=correct_headers, header=1)
        temp_df['structure_id'] = pdb_id
        hbond_list.append(temp_df)
    
    # Concatenate all individual PDB dataframes
    df_hbonds = pd.concat(hbond_list, ignore_index=True)
    
    # Extract the 3-letter Amino Acid residue name (e.g., LYS, ARG) 
    # This uses a regex to find the pattern '.XXXdigit' in the 'aa_atom' string
    df_hbonds['amino_acid_code'] = df_hbonds['aa_atom'].str.extract(r'\.([A-Z]{3})\d+')
    
    # Save to SQL so we can perform Joins with structural_proteins/structural_rna later
    df_hbonds.to_sql('hydrogen_bonds', conn, if_exists='replace', index=False)
    
    print("SUCCESS: 'hydrogen_bonds' table built. Headers standardized.")
    display(df_hbonds.head())

conn.close()

ERROR: No H-bond CSV files found in 'Project/'.


In [3]:
import pandas as pd
import glob
import os
import re

# 1. Target all parsed H-bond files
hbond_files = glob.glob('Project/*_hbonds_parsed.csv')
all_interactions = []

for f in hbond_files:
    pdb_id = os.path.basename(f).split('_')[0].upper()
    
    # Read file without headers to avoid the 'col1/col2' mismatch
    df = pd.read_csv(f, header=None)
    
    # 2. Search EVERY column for the protein pattern (e.g., .ARG, .LYS, .GLU)
    # We convert the whole row to a string and look for the dot + 3 capital letters
    mask = df.apply(lambda row: row.astype(str).str.contains(r'\.[A-Z]{3}', regex=True).any(), axis=1)
    df_prot = df[mask].copy()
    
    if not df_prot.empty:
        # 3. Extract the 3-letter code from the row
        def extract_from_row(row):
            combined_text = " ".join(row.astype(str))
            match = re.search(r'\.([A-Z]{3})', combined_text)
            return match.group(1) if match else None

        df_prot['amino_acid'] = df_prot.apply(extract_from_row, axis=1)
        df_prot['structure'] = pdb_id
        all_interactions.append(df_prot)

# 4. Final Aggregation
if all_interactions:
    df_master = pd.concat(all_interactions, ignore_index=True)
    stats = df_master.groupby(['structure', 'amino_acid']).size().reset_index(name='hbond_count')
    stats = stats.sort_values(by=['structure', 'hbond_count'], ascending=[True, False])
    
    print(f"SUCCESS: Found {len(df_master)} Protein-RNA interactions!")
    display(stats)
else:
    print("CRITICAL ERROR: No protein patterns found in any file. Let's check the raw text of one file:")
    with open('Project/1j5e_hbonds_parsed.csv', 'r') as file:
        print("".join(file.readlines()[:5]))

CRITICAL ERROR: No protein patterns found in any file. Let's check the raw text of one file:


FileNotFoundError: [Errno 2] No such file or directory: 'Project/1j5e_hbonds_parsed.csv'

In [12]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Prepare the data: Get the Top 5 amino acids for each structure
top_binders = stats.groupby('structure').head(5)

# 2. Create the Visualization (CS 210 Week 08)
plt.figure(figsize=(12, 6))
sns.barplot(data=top_binders, x='amino_acid', y='hbond_count', hue='structure', palette='viridis')

plt.title('Top 5 Protein Binders: Comparing Bacterial vs. Human Ribosomes', fontsize=14)
plt.xlabel('Amino Acid Residue', fontsize=12)
plt.ylabel('Total Hydrogen Bonds', fontsize=12)
plt.legend(title='Ribosome ID')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Save for your project report
plt.savefig('ribosome_comparison.png', dpi=300)
plt.show()

NameError: name 'stats' is not defined

In [ ]:
import pandas as pd
import os
from Bio.PDB import MMCIFParser

# Define target files
pdb_files = ['1J5E.cif', '1IBL.cif', '1HNW.cif', '6G5H.cif']
pdb_dir = 'Project/'
parser = MMCIFParser(QUIET=True)
all_structure_data = []

for file in pdb_files:
    file_path = os.path.join(pdb_dir, file)
    if os.path.exists(file_path):
        structure_id = file.split('.')[0]
        structure = parser.get_structure(structure_id, file_path)
        print(f"Initialized: {structure_id}")
        
        for model in structure:
            for chain in model:
                for residue in chain:
                    if residue.id[0] != ' ': continue # Skip heteroatoms
                    for atom in residue:
                        all_structure_data.append({
                            'structure_id': structure_id,
                            'residue_name': residue.resname,
                            'atom_name': atom.name,
                            'x_coord': atom.coord[0],
                            'y_coord': atom.coord[1],
                            'z_coord': atom.coord[2]
                        })

df_pdb_master = pd.DataFrame(all_structure_data)
print(f"Total atoms extracted: {len(df_pdb_master)}")
display(df_pdb_master.head())

In [ ]:
import glob
import re
hbond_files = glob.glob('*_hbonds_parsed.csv')
all_interactions = []
for f in hbond_files:
    pdb_id = os.path.basename(f).split('_')[0].upper()
    df = pd.read_csv(f, header=None)
    mask = df.apply(lambda row: row.astype(str).str.contains(r'\.[A-Z]{3}'), axis=1).any(axis=1)
    df_prot = df[mask].copy()
    if not df_prot.empty:
        def extract_code(row):
            match = re.search(r'\.([A-Z]{3})', " ".join(row.astype(str)))
            return match.group(1) if match else None
        df_prot['amino_acid'] = df_prot.apply(extract_code, axis=1)
        df_prot['structure'] = pdb_id
        all_interactions.append(df_prot)
df_master_bonds = pd.concat(all_interactions, ignore_index=True)
print(f"Successfully isolated {len(df_master_bonds)} Protein-RNA interactions")

In [ ]:
# Aggregate the counts
stats = df_master_bonds.groupby(['structure', 'amino_acid']).size().reset_index(name='hbond_count')
stats = stats.sort_values(by=['structure', 'hbond_count'], ascending=[True, False])

print("--- Final Interaction Counts ---")
display(stats.head(20))

# Optional: Print the specific champion for each structure
champions = stats.sort_values('hbond_count', ascending=False).drop_duplicates('structure')
print("\n--- Primary Anchor for Each Ribosome ---")
display(champions)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

top_5 = stats.groupby('structure').head(5)
plt.figure(figsize=(12, 6))
sns.barplot(data=top_5, x='amino_acid', y='hbond_count', hue='structure', palette='viridis')

plt.title('Top 5 Protein Binders: Bacterial vs. Human Ribosomes')
plt.ylabel('Total Hydrogen Bonds')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# 1. Get the RNA Base Counts (from your df_pdb_master)
# We filter for 1-letter residues and count them
rna_counts = df_pdb_master[df_pdb_master['residue_name'].str.len() == 1].copy()
rna_summary = rna_counts.groupby(['structure_id', 'residue_name']).size().reset_index(name='atom_count')

# 2. Get your Protein Bond Counts (from the 'stats' dataframe we just made)
# We'll rename the columns slightly to make the merge clean
protein_summary = stats.copy()
protein_summary.columns = ['structure_id', 'amino_acid', 'hbond_count']

# 3. Merge them into a single "Interface Table"
# This shows every RNA base and the primary protein binder associated with that structure
interface_table = pd.merge(rna_summary, protein_summary, on='structure_id')

# 4. Final Formatting: Pivot or Group to see the "Chemical Landscape"
# This view shows for each Structure: RNA Base | Base Atom Count | Amino Acid | Bond Count
print("--- Unified RNA-Protein Interface Table ---")
display(interface_table.head(20))

# OPTIONAL: A specific view showing the RNA 'G' vs Protein 'ARG' relationship
print("\n--- Focused Analysis: Guanine vs. Arginine Density ---")
g_arg_focus = interface_table[
    (interface_table['residue_name'] == 'G') & 
    (interface_table['amino_acid'] == 'ARG')
]
display(g_arg_focus)

In [ ]:
import pandas as pd

# 1. Ensure we have the RNA base extracted correctly
def extract_rna_base(val):
    match = re.search(r'\.([AUCG])', str(val))
    return match.group(1) if match else 'Unknown'

# Using the correct column index for nt_atom (usually column 1 or 'nt_atom')
# We'll use the column from our most recent df_master_bonds
df_master_bonds['rna_base'] = df_master_bonds.iloc[:, 1].apply(extract_rna_base)

# 2. Create the Pivot Table
pivot_table = pd.crosstab(df_master_bonds['amino_acid'], 
                          df_master_bonds['rna_base'], 
                          margins=True, 
                          margins_name="Grand_Total")

# 3. Fix the Sorting (Sort by the Grand_Total column we just named)
pivot_table = pivot_table.sort_values(by='Grand_Total', ascending=False)

# 4. Calculate Percentage (What % of ARG bonds are on G vs A?)
# This is "Data Normalization" - a key Week 06 skill
pivot_percent = pivot_table.div(pivot_table['Grand_Total'], axis=0) * 100

print("--- Count Matrix (Total Bonds) ---")
display(pivot_table)

print("\n--- Preference Matrix (% of Amino Acid's total bonds per Base) ---")
display(pivot_percent.round(2))

In [ ]:
import pandas as pd
import re

# 1. Broad Extraction: Just find the first A, U, C, or G in the string
def extract_rna_base_broad(val):
    val_str = str(val)
    # Search for A, U, C, or G. We look for these specifically 
    # to avoid accidentally grabbing letters from 'ARG' or 'ASN'
    match = re.search(r'[AUCG]', val_str)
    if match:
        return match.group(0)
    return 'Unknown'

# 2. Apply to column index 1 (the nt-atom column)
df_master_bonds['rna_base'] = df_master_bonds.iloc[:, 1].apply(extract_rna_base_broad)

# 3. Create the Pivot Table again
pivot_table = pd.crosstab(df_master_bonds['amino_acid'], 
                          df_master_bonds['rna_base'], 
                          margins=True, 
                          margins_name="Grand_Total")

# 4. Calculate Percentage Density
pivot_percent = pivot_table.div(pivot_table['Grand_Total'], axis=0) * 100

print("--- FINAL RECOVERY: Preference Matrix (%) ---")
# Dropping Grand_Total and Unknown to see the real biological distribution
display(pivot_percent.drop(columns=['Grand_Total', 'Unknown'], errors='ignore').round(2).head(15))

In [ ]:
import pandas as pd
import re

# 1. New Logic: Look for the pattern '@[AnyChain].[Base][Number]'
def extract_rna_base_final_v2(val):
    val_str = str(val)
    # This regex looks for:
    # 1. A dot .
    # 2. Followed by A, U, C, or G (The Base)
    # 3. Followed immediately by a number (\d)
    match = re.search(r'\.([AUCG])\d+', val_str)
    if match:
        return match.group(1)
    
    # Fallback: Just find the first A,U,C,G that isn't part of an atom name
    # We look at the end of the string specifically
    matches = re.findall(r'([AUCG])\d+', val_str)
    return matches[-1] if matches else 'Unknown'

# 2. Diagnostic: Show us what is actually in the column we are picking
target_col = 1 # We assume column index 1 is nt_atom
print("Sample values from target column:")
print(df_master_bonds.iloc[:, target_col].head(5).tolist())

# 3. Apply the fix
df_master_bonds['rna_base'] = df_master_bonds.iloc[:, target_col].apply(extract_rna_base_final_v2)

# 4. Create the Pivot Table
pivot_table = pd.crosstab(df_master_bonds['amino_acid'], 
                          df_master_bonds['rna_base'], 
                          margins=True, 
                          margins_name="Total")

# 5. Display Percentages (Dropping 'Unknown' to see the biological signal)
pivot_percent = pivot_table.div(pivot_table['Total'], axis=0) * 100
final_display = pivot_percent.drop(columns=['Total', 'Unknown'], errors='ignore')

print("\n--- BIOLOGICAL PREFERENCE MATRIX (%) ---")
if final_display.empty or final_display.shape[1] == 0:
    print("Error: Still only found 'Unknown'. Here is what the regex saw:")
    print(df_master_bonds['rna_base'].value_counts())
else:
    display(final_display.round(2).head(15))

In [45]:
import pandas as pd
import re

# 1. The most robust RNA extractor
def find_rna_base(val):
    val_str = str(val)
    # Look for a dot followed by A, U, C, or G and a number (e.g., .G9)
    match = re.search(r'\.([AUCG])\d+', val_str)
    if match:
        return match.group(1)
    return None

# 2. Automatically find which column has the RNA data
rna_col = None
for col in df_master_bonds.columns:
    # Check if this column has any values that look like RNA (e.g., .G9 or .A10)
    if df_master_bonds[col].astype(str).str.contains(r'\.[AUCG]\d+', regex=True).any():
        rna_col = col
        break

if rna_col is not None:
    print(f"Success! Found RNA data in column: {rna_col}")
    df_master_bonds['rna_base'] = df_master_bonds[rna_col].apply(find_rna_base)
    
    # 3. Create the Pivot Table
    pivot_table = pd.crosstab(df_master_bonds['amino_acid'], 
                              df_master_bonds['rna_base'], 
                              margins=True, 
                              margins_name="Total")
    
    # 4. Calculate Percentages
    pivot_percent = pivot_table.div(pivot_table['Total'], axis=0) * 100
    print("\n--- FINAL RNA PREFERENCE MATRIX (%) ---")
    display(pivot_percent.drop(columns=['Total'], errors='ignore').round(2).head(15))
else:
    print("Error: Could not find any columns containing RNA residue patterns (like .G9).")
    print("Current column headers are:", df_master_bonds.columns.tolist())
    display(df_master_bonds.head(3))

Success! Found RNA data in column: 2

--- FINAL RNA PREFERENCE MATRIX (%) ---


rna_base,A,C,G,U
amino_acid,,,,
ALA,30.23,4.65,39.53,25.58
ARG,21.18,29.67,32.58,16.57
ASN,25.54,28.26,24.46,21.74
ASP,11.76,19.61,45.10,23.53
CYS,71.43,0.00,14.29,14.29
GLN,21.05,17.54,41.52,19.88
GLU,25.93,19.75,39.51,14.81
GLY,24.12,20.00,37.06,18.82
HIS,18.90,27.56,33.07,20.47


In [46]:
import sqlite3
import pandas as pd

# Reconnect to the database you built
conn = sqlite3.connect('ribosome_data.db')

# Write a SQL query to pull all Hydrogen bonds for the human ribosome (6G5H)
sql_query = """
SELECT structure_id, nt_atom, aa_atom, distance, interaction_type
FROM hydrogen_bonds
WHERE structure_id = '6G5H'
ORDER BY distance ASC
LIMIT 10;
"""

# Execute and display
shortest_bonds_6g5h = pd.read_sql_query(sql_query, conn)
display(shortest_bonds_6g5h)

conn.close()

DatabaseError: Execution failed on sql '
SELECT structure_id, nt_atom, aa_atom, distance, interaction_type
FROM hydrogen_bonds
WHERE structure_id = '6G5H'
ORDER BY distance ASC
LIMIT 10;
': no such table: hydrogen_bonds